# 04 - Comprehensive Model Evaluation

This notebook provides a thorough evaluation of our fine-tuned BioBERT NER model.

**Goals:**
1. Entity-level metrics (per-entity precision/recall/F1)
2. Test across different input categories
3. Confidence score analysis
4. Error analysis and model limitations
5. Performance benchmarking

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

import torch
import json
import time
from collections import defaultdict
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 1. Load the Fine-Tuned Model

In [ ]:
# Find the latest trained model
model_dir = project_root / "outputs" / "models"
model_runs = sorted(model_dir.glob("run_*"))

if model_runs:
    latest_model = model_runs[-1] / "final_model"
    print(f"Using model: {latest_model}")
else:
    raise FileNotFoundError("No trained models found. Run train.py first!")

# Load training info
training_info_path = model_runs[-1] / "training_info.json"
if training_info_path.exists():
    with open(training_info_path) as f:
        training_info = json.load(f)
    print(f"\nTraining Info:")
    print(f"  Model: {training_info.get('model_name', 'N/A')}")
    print(f"  Train samples: {training_info.get('train_samples', 'N/A')}")
    print(f"  Epochs: {training_info.get('epochs', 'N/A')}")

In [ ]:
# Import and initialize predictor
from src.predict import NERPredictor

print("Loading fine-tuned model...")
predictor = NERPredictor(str(latest_model))
print(f"Model loaded successfully!")
print(f"Labels: {predictor.labels}")

## 2. Define Test Categories

We'll test the model across different input styles and complexity levels.

In [ ]:
# Comprehensive test cases organized by category
TEST_CATEGORIES = {
    "simple": [
        "I have a headache.",
        "My throat hurts.",
        "I feel dizzy.",
        "I have a fever.",
        "My back hurts.",
    ],
    "complex": [
        "Patient presents with bilateral pneumonia and acute respiratory distress.",
        "I've been experiencing persistent chest pain radiating to my left arm.",
        "She has a history of chronic migraines with aura and photophobia.",
        "The patient shows signs of diabetic ketoacidosis with polyuria and polydipsia.",
        "Experiencing severe abdominal pain with nausea, vomiting, and diarrhea.",
    ],
    "casual": [
        "my stomach's been killing me lately",
        "cant stop coughing, its so annoying",
        "feeling super tired all the time",
        "got this weird rash on my arm",
        "head is pounding like crazy",
    ],
    "medical_jargon": [
        "Diagnosed with T2DM, currently on metformin 500mg BID.",
        "Patient has HTN and hyperlipidemia, on lisinopril and atorvastatin.",
        "Presenting with SOB and DOE, possible CHF exacerbation.",
        "History of GERD, currently taking omeprazole PRN.",
        "Patient c/o N/V x 3 days with mild epigastric tenderness.",
    ],
    "medications": [
        "I took ibuprofen for my headache but it didn't help.",
        "Currently taking aspirin, metformin, and lisinopril daily.",
        "The doctor prescribed amoxicillin for my infection.",
        "I've been on prednisone for my asthma flare-up.",
        "Started acetaminophen for fever, considering adding ibuprofen.",
    ],
    "conditions": [
        "I have diabetes and high blood pressure.",
        "My asthma has been acting up recently.",
        "I was diagnosed with hypertension last year.",
        "Living with rheumatoid arthritis for 5 years.",
        "My depression and anxiety have been worse lately.",
    ],
    "edge_cases": [
        "I just don't feel well.",
        "Something is wrong but I can't explain it.",
        "I feel off today.",
        "",  # Empty string
        "The weather is nice today.",  # No medical content
    ],
    "multi_symptom": [
        "I have a high fever, severe headache, stiff neck, and sensitivity to light.",
        "Experiencing fatigue, weight loss, increased thirst, and frequent urination.",
        "Chest pain, shortness of breath, dizziness, and nausea all at once.",
        "Runny nose, sore throat, cough, body aches, and chills.",
        "Joint pain, swelling, morning stiffness, and fatigue in both hands.",
    ],
}

# Count total tests
total_tests = sum(len(tests) for tests in TEST_CATEGORIES.values())
print(f"Total test cases: {total_tests}")
print(f"Categories: {list(TEST_CATEGORIES.keys())}")

## 3. Run Comprehensive Evaluation

In [ ]:
def evaluate_text(text, predictor):
    """Evaluate a single text and return detailed results."""
    start_time = time.time()
    
    if not text.strip():
        return {
            'text': text,
            'entities': [],
            'entity_count': 0,
            'inference_time': 0,
            'by_type': {}
        }
    
    result = predictor.predict(text)
    inference_time = time.time() - start_time
    
    entities = result.entities if hasattr(result, 'entities') else result.get('entities', [])
    
    # Group by entity type
    by_type = defaultdict(list)
    for ent in entities:
        if hasattr(ent, 'label'):
            by_type[ent.label].append({
                'text': ent.text,
                'confidence': ent.confidence,
                'span': ent.span
            })
        else:
            by_type[ent.get('label', 'Unknown')].append(ent)
    
    return {
        'text': text,
        'entities': entities,
        'entity_count': len(entities),
        'inference_time': inference_time,
        'by_type': dict(by_type)
    }

In [ ]:
# Run evaluation on all categories
print("Running comprehensive evaluation...\n")
print("=" * 80)

all_results = {}
all_confidences = []
entity_type_counts = defaultdict(int)
category_stats = {}

for category, tests in TEST_CATEGORIES.items():
    print(f"\n### Category: {category.upper()} ###")
    print("-" * 60)
    
    category_results = []
    category_entities = 0
    category_times = []
    
    for text in tests:
        result = evaluate_text(text, predictor)
        category_results.append(result)
        category_entities += result['entity_count']
        category_times.append(result['inference_time'])
        
        # Collect confidences
        for ent in result['entities']:
            conf = ent.confidence if hasattr(ent, 'confidence') else ent.get('confidence', 0)
            all_confidences.append(conf)
            label = ent.label if hasattr(ent, 'label') else ent.get('label', 'Unknown')
            entity_type_counts[label] += 1
        
        # Print result
        display_text = text[:60] + "..." if len(text) > 60 else text
        print(f"\nInput: {display_text}")
        if result['entities']:
            for ent in result['entities']:
                ent_text = ent.text if hasattr(ent, 'text') else ent.get('text', '')
                ent_label = ent.label if hasattr(ent, 'label') else ent.get('label', '')
                ent_conf = ent.confidence if hasattr(ent, 'confidence') else ent.get('confidence', 0)
                print(f"  -> {ent_text}: {ent_label} ({ent_conf:.1%})")
        else:
            print("  -> No entities detected")
    
    all_results[category] = category_results
    category_stats[category] = {
        'total_entities': category_entities,
        'avg_entities': category_entities / len(tests),
        'avg_time': np.mean(category_times) * 1000,  # ms
        'tests': len(tests)
    }

print("\n" + "=" * 80)

## 4. Category-Level Analysis

In [ ]:
# Create summary DataFrame
summary_data = []
for category, stats in category_stats.items():
    summary_data.append({
        'Category': category,
        'Test Cases': stats['tests'],
        'Total Entities': stats['total_entities'],
        'Avg Entities/Input': f"{stats['avg_entities']:.2f}",
        'Avg Inference (ms)': f"{stats['avg_time']:.1f}"
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "=" * 60)
print("CATEGORY SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

In [ ]:
# Visualize entities by category
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Entities per category
categories = list(category_stats.keys())
entity_counts = [category_stats[c]['total_entities'] for c in categories]

axes[0].bar(categories, entity_counts, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Total Entities Detected')
axes[0].set_title('Entities Detected by Category')
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Entity type distribution
entity_types = list(entity_type_counts.keys())
type_counts = list(entity_type_counts.values())

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
axes[1].pie(type_counts, labels=entity_types, autopct='%1.1f%%', colors=colors[:len(entity_types)])
axes[1].set_title('Entity Type Distribution')

plt.tight_layout()
plt.show()

## 5. Confidence Score Analysis

In [ ]:
if all_confidences:
    confidences = np.array(all_confidences)
    
    print("\n" + "=" * 60)
    print("CONFIDENCE SCORE ANALYSIS")
    print("=" * 60)
    print(f"Total predictions: {len(confidences)}")
    print(f"Mean confidence: {np.mean(confidences):.2%}")
    print(f"Median confidence: {np.median(confidences):.2%}")
    print(f"Std deviation: {np.std(confidences):.2%}")
    print(f"Min confidence: {np.min(confidences):.2%}")
    print(f"Max confidence: {np.max(confidences):.2%}")
    
    # Confidence brackets
    print(f"\nConfidence Distribution:")
    print(f"  High (>90%): {np.sum(confidences > 0.9)} ({np.sum(confidences > 0.9)/len(confidences):.1%})")
    print(f"  Medium (70-90%): {np.sum((confidences >= 0.7) & (confidences <= 0.9))} ({np.sum((confidences >= 0.7) & (confidences <= 0.9))/len(confidences):.1%})")
    print(f"  Low (<70%): {np.sum(confidences < 0.7)} ({np.sum(confidences < 0.7)/len(confidences):.1%})")
else:
    print("No predictions to analyze.")

In [ ]:
# Visualize confidence distribution
if all_confidences:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    axes[0].hist(confidences, bins=20, color='steelblue', alpha=0.7, edgecolor='black')
    axes[0].axvline(x=np.mean(confidences), color='red', linestyle='--', label=f'Mean: {np.mean(confidences):.2%}')
    axes[0].set_xlabel('Confidence Score')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Confidence Score Distribution')
    axes[0].legend()
    
    # Box plot
    axes[1].boxplot(confidences, vert=True)
    axes[1].set_ylabel('Confidence Score')
    axes[1].set_title('Confidence Score Box Plot')
    
    plt.tight_layout()
    plt.show()

## 6. Error Analysis

In [ ]:
# Identify potential issues
print("\n" + "=" * 60)
print("ERROR ANALYSIS")
print("=" * 60)

# 1. Inputs with no entities detected (excluding edge cases)
print("\n1. Inputs with NO entities detected (excluding edge cases):")
no_entity_inputs = []
for category, results in all_results.items():
    if category != 'edge_cases':
        for result in results:
            if result['entity_count'] == 0 and result['text'].strip():
                no_entity_inputs.append((category, result['text']))

if no_entity_inputs:
    for cat, text in no_entity_inputs:
        print(f"  [{cat}] {text}")
else:
    print("  None - all inputs had at least one entity detected!")

# 2. Low confidence predictions
print("\n2. Low confidence predictions (<50%):")
low_conf_predictions = []
for category, results in all_results.items():
    for result in results:
        for ent in result['entities']:
            conf = ent.confidence if hasattr(ent, 'confidence') else ent.get('confidence', 0)
            if conf < 0.5:
                ent_text = ent.text if hasattr(ent, 'text') else ent.get('text', '')
                ent_label = ent.label if hasattr(ent, 'label') else ent.get('label', '')
                low_conf_predictions.append((result['text'][:40], ent_text, ent_label, conf))

if low_conf_predictions:
    for text, ent_text, label, conf in low_conf_predictions[:10]:  # Show first 10
        print(f"  '{ent_text}' ({label}) - {conf:.1%} in '{text}...'")
else:
    print("  None - all predictions have confidence >= 50%!")

In [ ]:
# 3. Analyze edge cases
print("\n3. Edge Case Analysis:")
print("-" * 40)
for result in all_results.get('edge_cases', []):
    text = result['text'] if result['text'] else "[EMPTY STRING]"
    entities = result['entity_count']
    print(f"  Input: {text}")
    print(f"  Entities found: {entities}")
    if result['entities']:
        for ent in result['entities']:
            ent_text = ent.text if hasattr(ent, 'text') else ent.get('text', '')
            ent_label = ent.label if hasattr(ent, 'label') else ent.get('label', '')
            print(f"    -> {ent_text}: {ent_label}")
    print()

## 7. Performance Benchmarking

In [ ]:
# Benchmark inference speed
print("\n" + "=" * 60)
print("PERFORMANCE BENCHMARKING")
print("=" * 60)

# Collect all inference times
all_times = []
for category, results in all_results.items():
    for result in results:
        if result['inference_time'] > 0:
            all_times.append(result['inference_time'])

if all_times:
    times_ms = np.array(all_times) * 1000
    print(f"\nInference Time Statistics:")
    print(f"  Total predictions: {len(times_ms)}")
    print(f"  Mean: {np.mean(times_ms):.2f} ms")
    print(f"  Median: {np.median(times_ms):.2f} ms")
    print(f"  Min: {np.min(times_ms):.2f} ms")
    print(f"  Max: {np.max(times_ms):.2f} ms")
    print(f"  Throughput: {1000/np.mean(times_ms):.1f} predictions/second")

## 8. Model Capabilities Summary

In [ ]:
# Final summary
print("\n" + "#" * 60)
print("#" + " MODEL CAPABILITIES SUMMARY ".center(58) + "#")
print("#" * 60)

total_entities = sum(category_stats[c]['total_entities'] for c in category_stats)
total_tests = sum(category_stats[c]['tests'] for c in category_stats)

print(f"\n  Model: {training_info.get('model_name', 'BioBERT')}")
print(f"  Test cases evaluated: {total_tests}")
print(f"  Total entities detected: {total_entities}")
print(f"  Average entities per input: {total_entities/total_tests:.2f}")

if all_confidences:
    print(f"\n  Confidence Metrics:")
    print(f"    Mean: {np.mean(all_confidences):.1%}")
    print(f"    High confidence (>90%): {np.sum(np.array(all_confidences) > 0.9)/len(all_confidences):.1%}")

print(f"\n  Strengths:")
print(f"    - Detects medical symptoms, diseases, and chemicals")
print(f"    - Handles both formal and casual language")
print(f"    - Fast inference (~{np.mean(times_ms):.0f}ms per prediction)")

print(f"\n  Limitations:")
print(f"    - May miss rare medical abbreviations")
print(f"    - Subword tokenization can split entities")
print(f"    - Trained on limited dataset (180 samples)")

print("\n" + "#" * 60)

## 9. Save Evaluation Results

In [ ]:
# Save comprehensive results
evaluation_results = {
    'timestamp': datetime.now().isoformat(),
    'model_path': str(latest_model),
    'total_tests': total_tests,
    'total_entities': total_entities,
    'category_stats': category_stats,
    'entity_type_distribution': dict(entity_type_counts),
    'confidence_stats': {
        'mean': float(np.mean(all_confidences)) if all_confidences else 0,
        'median': float(np.median(all_confidences)) if all_confidences else 0,
        'std': float(np.std(all_confidences)) if all_confidences else 0,
        'min': float(np.min(all_confidences)) if all_confidences else 0,
        'max': float(np.max(all_confidences)) if all_confidences else 0,
    },
    'performance': {
        'mean_inference_ms': float(np.mean(times_ms)) if all_times else 0,
        'throughput_per_second': float(1000/np.mean(times_ms)) if all_times else 0,
    }
}

# Save to results directory
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

with open(results_dir / 'comprehensive_evaluation.json', 'w') as f:
    json.dump(evaluation_results, f, indent=2)

print(f"Results saved to results/comprehensive_evaluation.json")